# Multi-dataset LSTM: FD002, FD003, FD004
Tests whether the FD001 architecture generalises to multi-condition (FD002, FD004) and dual-fault (FD003, FD004) datasets. Runs sequentially; skips any dataset whose weights file already exists in Drive, so re-running after a runtime reset is safe.

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE  = '/content/drive/MyDrive/engine-data-project'
DATA   = f'{DRIVE}/data'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
COLS = (
    ['engine_id', 'cycle']
    + [f'op_setting_{i}' for i in range(1, 4)]
    + [f'sensor_{i}'    for i in range(1, 22)]
)

WINDOW_SIZE    = 30
STD_THRESHOLD  = 0.05
MAX_EPOCHS     = 50
EARLY_STOP_PAT = 10
LR             = 0.0005
BATCH_TRAIN    = 256
BATCH_VAL      = 512

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LSTMRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.lstm1 = nn.LSTM(n_features, 64, batch_first=True)
        self.drop1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(64, 32, batch_first=True)
        self.drop2 = nn.Dropout(0.2)
        self.fc    = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out    = self.drop1(out)
        out, _ = self.lstm2(out)
        # only the last timestep carries accumulated sequence context
        out    = self.drop2(out[:, -1, :])
        return self.fc(out).squeeze(1)


def nasa_score(y_true, y_pred):
    d = np.array(y_pred) - np.array(y_true)
    # late predictions penalised harder than early; asymmetric cost from NASA spec
    penalties = np.where(d < 0, np.expm1(-d / 13), np.expm1(d / 10))
    return penalties.sum()

In [ ]:
def load_dataset(fd_name):
    df = pd.read_csv(
        f'{DATA}/train_{fd_name}.txt',
        sep=r'\s+', header=None, names=COLS, engine='python'
    )

    max_cycle = df.groupby('engine_id')['cycle'].max().rename('max_cycle')
    df = df.join(max_cycle, on='engine_id')
    df['rul'] = (df['max_cycle'] - df['cycle']).clip(upper=125)
    df = df.drop(columns='max_cycle')

    feature_cols = [c for c in df.columns if c not in ('engine_id', 'cycle', 'rul')]
    stds         = df[feature_cols].std()
    drop_cols    = stds[stds < STD_THRESHOLD].index.tolist()
    keep_cols    = [c for c in feature_cols if c not in drop_cols]
    df           = df.drop(columns=drop_cols)
    print(f'  {fd_name}: {len(drop_cols)} cols dropped, {len(keep_cols)} features kept')

    def _scale(group):
        g   = group.copy()
        mn  = g[keep_cols].min()
        mx  = g[keep_cols].max()
        rng = mx - mn
        rng[rng == 0] = 1  # flat feature within one engine; avoids NaN from divide-by-zero
        g[keep_cols] = (g[keep_cols] - mn) / rng
        return g

    df = df.groupby('engine_id', group_keys=False).apply(_scale)
    return df, keep_cols

In [ ]:
def build_windows(df, sensor_cols):
    windows, labels, eids = [], [], []
    for eid, group in df.groupby('engine_id'):
        group = group.sort_values('cycle')
        s = group[sensor_cols].values
        r = group['rul'].values
        # engines shorter than WINDOW_SIZE yield no windows
        for i in range(len(group) - WINDOW_SIZE + 1):
            windows.append(s[i : i + WINDOW_SIZE])
            labels.append(r[i + WINDOW_SIZE - 1])
            eids.append(eid)
    return (
        np.array(windows, dtype=np.float32),
        np.array(labels,  dtype=np.float32),
        np.array(eids),
    )

In [ ]:
def run_training(X_train, y_train, X_val, y_val):
    n_features   = X_train.shape[2]
    model        = LSTMRegressor(n_features).to(device)
    optimizer    = torch.optim.Adam(model.parameters(), lr=LR)
    criterion    = nn.MSELoss()
    scheduler    = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=0.5, patience=5
    )

    train_loader = DataLoader(
        WindowDataset(X_train, y_train), batch_size=BATCH_TRAIN, shuffle=True
    )
    val_loader = DataLoader(
        WindowDataset(X_val, y_val), batch_size=BATCH_VAL, shuffle=False
    )

    best_val_loss  = float('inf')
    best_weights   = None
    patience_count = 0
    stopped_epoch  = MAX_EPOCHS

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            # 30-step sequences accumulate gradients; clipping prevents LSTM gate blowup
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(yb)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item() * len(yb)
        val_loss /= len(val_loader.dataset)

        scheduler.step(val_loss)
        print(
            f'  epoch {epoch:02d}  train {train_loss:.4f}  val {val_loss:.4f}'
            f'  lr {optimizer.param_groups[0]["lr"]:.6f}'
        )

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            best_weights   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= EARLY_STOP_PAT:
                stopped_epoch = epoch
                print(f'  early stop at epoch {epoch}')
                break

    model.load_state_dict(best_weights)
    return model, stopped_epoch

In [ ]:
def evaluate(model, X_val, y_val):
    loader = DataLoader(WindowDataset(X_val, y_val), batch_size=BATCH_VAL, shuffle=False)
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            preds.append(model(xb.to(device)).cpu().numpy())
    preds = np.concatenate(preds)
    rmse  = float(np.sqrt(((preds - y_val) ** 2).mean()))
    score = float(nasa_score(y_val, preds))
    return rmse, score

In [ ]:
DATASETS    = ['FD002', 'FD003', 'FD004']
all_results = {}

for fd in DATASETS:
    weight_path = f'{DRIVE}/lstm_{fd.lower()}.pth'
    result_path = f'{DRIVE}/results_{fd.lower()}.json'

    if os.path.exists(weight_path):
        print(f'{fd}: weights already saved, skipping')
        if os.path.exists(result_path):
            with open(result_path) as fh:
                all_results[fd] = json.load(fh)
        continue

    print(f'\n=== {fd} ===')

    df, sensor_cols = load_dataset(fd)

    engine_ids  = sorted(df['engine_id'].unique())
    n_val       = max(1, int(len(engine_ids) * 0.2))
    val_engines = set(engine_ids[-n_val:])

    X, y, win_eids = build_windows(df, sensor_cols)

    train_mask = ~np.isin(win_eids, list(val_engines))
    val_mask   =  np.isin(win_eids, list(val_engines))
    X_train, y_train = X[train_mask], y[train_mask]
    X_val,   y_val   = X[val_mask],   y[val_mask]
    print(f'  train: {X_train.shape}  val: {X_val.shape}')

    model, stopped_epoch = run_training(X_train, y_train, X_val, y_val)
    rmse, score          = evaluate(model, X_val, y_val)

    torch.save(model.state_dict(), weight_path)
    print(f'  weights saved to {weight_path}')

    result = {
        'RMSE':          round(rmse, 2),
        'NASA_score':    int(round(score)),
        'n_engines':     len(engine_ids),
        'n_sensors':     len(sensor_cols),
        'epochs_trained': stopped_epoch,
    }
    with open(result_path, 'w') as fh:
        json.dump(result, fh, indent=2)

    all_results[fd] = result
    print(f'{fd} done. RMSE {rmse:.2f} NASA {int(round(score))}')

In [ ]:
# pick up results written in a prior session before a runtime reset
for fd in DATASETS:
    if fd not in all_results:
        rp = f'{DRIVE}/results_{fd.lower()}.json'
        if os.path.exists(rp):
            with open(rp) as fh:
                all_results[fd] = json.load(fh)

hdr = f"{'Dataset':<10} {'RMSE':>8} {'NASA Score':>12} {'Epochs':>8}"
print(hdr)
print('-' * len(hdr))
print(f"{'FD001':<10} {20.83:>8.2f} {19490:>12} {'(nb04)':>8}")
for fd in DATASETS:
    if fd in all_results:
        r = all_results[fd]
        print(f"{fd:<10} {r['RMSE']:>8.2f} {r['NASA_score']:>12} {r['epochs_trained']:>8}")
    else:
        print(f"{fd:<10} {'N/A':>8} {'N/A':>12} {'N/A':>8}")